Tạo Spark Session:

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("FeatureEngineering")\
    .config("spark.driver.memory", "4g")\
    .config("spark.sql.shuffle.partitions", "8")\
    .getOrCreate()

Load datasets:


In [2]:
df_orders = spark.read.csv(path= "/home/jovyan/data/orders.csv", header= True, inferSchema= True)
# df_prior = spark.read.csv(path="/home/jovyan/data/order_products__prior.csv", header= True, inferSchema= True)
df_prior = spark.read.csv("/home/jovyan/data/order_products__prior.csv", header=True, inferSchema=True).sample(fraction=0.1, seed=42)
df_train = spark.read.csv(path="/home/jovyan/data/order_products__train.csv", header= True, inferSchema= True)
df_products = spark.read.csv(path="/home/jovyan/data/products.csv", header= True, inferSchema= True)
df_aisles = spark.read.csv(path="/home/jovyan/data/aisles.csv", header= True, inferSchema= True)
df_departments = spark.read.csv(path="/home/jovyan/data/departments.csv", header= True, inferSchema= True)

df_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)



Xem các cột và sửa giống như phần exploration:
df_train: order cuối cùng mà các người dùng yêu cầu
df_prior: lịch sử các order của người dùng từ trước đến nay

In [ ]:
print(f'order: {df_orders.printSchema()}')
print(f'aisles: {df_aisles.printSchema()}')
print(f'department: {df_departments.printSchema()}')
print(f'product: {df_products.printSchema()}')
print(f'train: {df_train.printSchema()}')
print(f'prior: {df_prior.printSchema()}')

In [4]:
from pyspark.sql.functions import col
df_products = df_products.withColumn('department_id', col('department_id').cast('integer'))
df_products = df_products.withColumn('aisle_id', col('aisle_id').cast('integer'))

In [ ]:
# Quick schema reference
print("=== ORDERS ===");        df_orders.printSchema()
print("=== PRIOR ===");         df_prior.printSchema()
print("=== TRAIN ===");         df_train.printSchema()
print("=== PRODUCTS ===");      df_products.printSchema()

Đặc trưng của user:
Người mua sẽ có các đặc trưng cần cân nhắc: 
'''
Số lượng đồ trong mỗi hóa đơn(basket_size)
Khoảng cách giữa các đợt mua hàng(avg_days_between_orders)
Người này đã có bao nhiêu lần mua(total_orders)
Trung bình đặt hàng lại(mean of reordered)
'''

In [6]:
from pyspark.sql.functions import count, mean, countDistinct, max

df_orders_prior = df_orders.join(df_prior, on= 'order_id', how= 'left').cache()

user_features = df_orders.groupBy('user_id').agg(
    count('order_id').alias('total_orders'),
    mean('days_since_prior_order'). alias('avg_days_between_orders')

)

basket_per_order = df_orders_prior.groupBy('user_id', 'order_id').agg(
    count('product_id').alias('basket_size')
)

user_basket = basket_per_order.groupBy('user_id').agg(
    mean('basket_size').alias('avg_basket_size')
)
user_reorder = df_orders_prior.groupBy('user_id').agg(
    mean('reordered').alias('overall_reorder_rate')
)

user_features = user_features.join(user_basket, on= 'user_id', how= 'left').join(user_reorder, on= 'user_id', how= 'left')

Đặc trung cho products:
Tổng số lần xuất hiện của mặt hàng trong hóa đơn
Trung bình số lần sản phẩm này được reordered
Sản phẩm này có bao nhiêu người mua khác nhau

In [7]:
product_features = df_prior.groupBy('product_id').agg(
    count('order_id').alias('product_total_orders')
)

product_reorder_rate_users = df_orders_prior.groupBy('product_id').agg(
    mean('reordered').alias('reorder_rate'),
    countDistinct('user_id').alias('unique_user')
)

product_features = product_features.join(product_reorder_rate_users, on= 'product_id', how= 'left')

Đặc trưng cho user-product:

In [8]:
up_features = df_orders_prior.groupBy('user_id', 'product_id').agg(
    count('order_id').alias('up_times_bought'),
    mean('reordered').alias('up_reorder_rate'),
    mean('add_to_cart_order').alias('up_avg_cart_pos'),
    max('order_number').alias('up_last_order')
)

Trong mỗi cặp user-product: 
Có bao nhiêu lần cặp này xuất hiện
Tỉ lệ mua lại sản phẩm là bao nhiêu
Vị trí của sản phẩm trong giỏ hàng
Lần gần nhất mà người này mua sản phẩm này

In [9]:
df_unique_prior = df_orders_prior.select('user_id', 'product_id')
df_unique_prior = df_unique_prior.dropDuplicates()

df_orders_train = df_train.join(df_orders, on= 'order_id', how= 'left')
df_unique_train = df_orders_train.select('user_id', 'product_id')
df_unique_train = df_unique_train.dropDuplicates()


from pyspark.sql.functions import when, lit
df_unique_train = df_unique_train.withColumn('label', lit(1))
df_unique = df_unique_prior.join(df_unique_train, on= ['product_id', 'user_id'], how='left')
df_unique = df_unique.withColumn('label', when(col('label').isNull(), 0).otherwise(1))

Sau khi ghép 2 bảng là orders và prior, chọn ra các cặp user-product khác nhau và loại bỏ lặp

Tương tự với orders và train, lần này sẽ cho ra các cặp user-product trong last order

Kết hợp 2 bảng này thì cặp nào có dữ liệu merge thì label là 1, ngược lại là 0

In [ ]:
df_unique = df_unique.join(user_features, on='user_id', how='left') \
    .join(product_features, on='product_id', how='left') \
    .join(up_features, on=['product_id', 'user_id'], how='left')


df_unique.show() 

In [32]:
from pyspark.sql.functions import count, mean, row_number
from pyspark.sql.window import Window

df_orders_prior = df_orders.join(df_prior, on='order_id', how='left')

user_features = df_orders.groupBy('user_id').agg(
    count('order_id').alias('total_orders'),
    mean('days_since_prior_order').alias('avg_days_between_orders')
)

basket_per_order = df_orders_prior.groupBy('user_id', 'order_id').agg(
    count('product_id').alias('basket_size')
)

user_basket = basket_per_order.groupBy('user_id').agg(
    mean('basket_size').alias('avg_basket_size')
)

user_reorder = df_orders_prior.groupBy('user_id').agg(
    mean('reordered').alias('overall_reorder_rate')
)

user_hour = df_orders.groupBy('user_id', 'order_hour_of_day') \
    .agg(count('order_id').alias('hour_count'))

window = Window.partitionBy('user_id').orderBy(col('hour_count').desc())
user_hour = user_hour.withColumn('rank', row_number().over(window))
favorite_hour = user_hour.filter(col('rank') == 1).select('user_id', col('order_hour_of_day').alias('favorite_hour'))

user_dow = df_orders.groupBy('user_id', 'order_dow') \
    .agg(count('order_id').alias('day_count'))
window = Window.partitionBy('user_id').orderBy(col('day_count').desc())
user_day = user_dow.withColumn('rank',row_number().over(window))
favorite_day = user_day.filter(col('rank') == 1).select('user_id', col('order_dow').alias('favorite_day'))

user_features = user_features \
    .join(user_basket, on='user_id', how='left') \
    .join(user_reorder, on='user_id', how='left') \
    .join(favorite_hour, on='user_id', how='left') \
    .join(favorite_day, on='user_id', how='left')

user_features.write.parquet('/home/jovyan/data/features/user_features',mode='overwrite')



In [14]:
from pyspark.sql.functions import countDistinct

product_features = df_orders_prior.groupBy('product_id').agg(
    count('order_id').alias('product_total_orders'),
    mean('reordered').alias('product_reorder_rate'),
    countDistinct('user_id').alias('product_unique_users')
)

product_features.write.parquet('/home/jovyan/data/features/product_features', mode = 'overwrite')

In [34]:
df_orders_prior_products = df_orders_prior.join(df_products, on='product_id', how='left')

background_product_features = df_orders_prior_products.groupBy('user_id', 'department_id').agg(
    count('order_id').alias('department_count')
)
window = Window.partitionBy('user_id').orderBy(col('department_count').desc())
product_department = background_product_features.withColumn('rank', row_number().over(window))
favorite_department = product_department.filter(col('rank') == 1).select('user_id', col('department_id').alias('favorite_department'))


aisles_features = df_orders_prior_products.groupBy('user_id', 'aisle_id').agg(
    count('order_id').alias('aisles_count')
)
window = Window.partitionBy('user_id').orderBy(col('aisles_count').desc())
product_aisles = aisles_features.withColumn('rank', row_number().over(window))
favorite_aisles = product_aisles.filter(col('rank') == 1).select('user_id', col('aisle_id').alias('favorite_aisles'))

In [35]:
user_features = user_features \
    .join(favorite_department, on='user_id', how='left') \
    .join(favorite_aisles, on='user_id', how='left')


user_features.write.parquet('/home/jovyan/data/features/user_features',mode='overwrite')

In [ ]:
user_features.columns

In [38]:
from pyspark.sql.functions import max
up_features = df_orders_prior.groupBy('user_id', 'product_id').agg(
    count('order_id').alias('up_times_bought'),
    mean('reordered').alias('up_reorder_rate'),
    mean('add_to_cart_order').alias('up_avg_cart_pos'),
    max('order_number').alias('up_last_order')
)

up_features = up_features.join(user_features.select('user_id', 'total_orders'), on='user_id', how='left')
up_features = up_features.withColumn('orders_since_last_buy', col('total_orders') - col('up_last_order'))
up_features = up_features.withColumn('up_order_rate', col('up_times_bought') / col('total_orders'))
up_features = up_features.drop('total_orders')
up_features.write.parquet('/home/jovyan/data/features/up_features', mode='overwrite')

In [ ]:
up_features.columns

In [41]:
from pyspark.sql.functions import when, lit, col

# Base table
df_unique_prior = df_orders_prior.select('user_id', 'product_id').dropDuplicates()

# Train labels
df_orders_train = df_train.join(df_orders, on='order_id', how='left')
df_unique_train = df_orders_train.select('user_id', 'product_id').dropDuplicates()
df_unique_train = df_unique_train.withColumn('label', lit(1))

# Join to create labels
df_base = df_unique_prior.join(df_unique_train, on=['user_id', 'product_id'], how='left')
df_base = df_base.withColumn('label', when(col('label').isNull(), 0).otherwise(1))

# Load saved features
user_features = spark.read.parquet('/home/jovyan/data/features/user_features')
product_features = spark.read.parquet('/home/jovyan/data/features/product_features')
up_features = spark.read.parquet('/home/jovyan/data/features/up_features')

# Join all features
df_final = df_base \
    .join(user_features, on='user_id', how='left') \
    .join(product_features, on='product_id', how='left') \
    .join(up_features, on=['user_id', 'product_id'], how='left')

df_final.write.parquet('/home/jovyan/data/features/final_dataset', mode='overwrite')

In [ ]:
df_final = spark.read.parquet('/home/jovyan/data/features/final_dataset')
df_final.printSchema()
df_final.count()

In [ ]:
print('user_features:', user_features.count())
print('product_features:', product_features.count())
print('up_features:', up_features.count())
print('df_base:', df_base.count())

In [ ]:
user_features.count() - user_features.dropDuplicates(['user_id']).count()

In [ ]:
df_orders_prior.groupBy('user_id') \
    .agg(mean('reordered').alias('reorder_rate')) \
    .filter(col('reorder_rate').isNull()) \
    .show()